In [ ]:
# ============================================================
# EXERCISE 7
# BOLTZMANN AND CAUCHY MACHINES
# STOCHASTIC FEATURE SELECTION AND ENERGY OPTIMIZATION
# USING WINE QUALITY DATASET
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# ============================================================
# 1. LOAD WINEQT DATASET
# ============================================================

df = pd.read_csv("WineQT.csv")

print("Dataset loaded successfully!")

print("\nDataset Shape:")
print(df.shape)

print("\nAvailable Columns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())


# ============================================================
# 2. SEPARATE FEATURES AND TARGET
# ============================================================

target_column = "quality"

# Check target column
if target_column not in df.columns:

    print("\nERROR: 'quality' column not found.")

    print("Available columns:")
    print(df.columns.tolist())

    raise ValueError(
        "The target column 'quality' is missing."
    )


# Separate features and target
X = df.drop(columns=[target_column])
y = df[target_column]

# Remove ID column if present
if "Id" in X.columns:
    X = X.drop(columns=["Id"])

if "id" in X.columns:
    X = X.drop(columns=["id"])


feature_names = X.columns.tolist()


print("\nNumber of Features:", len(feature_names))

print("\nFeatures:")

for i, feature in enumerate(feature_names):

    print(i, "->", feature)


# ============================================================
# 3. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("\nTraining Samples:", len(X_train))
print("Testing Samples :", len(X_test))


# ============================================================
# 4. CLASSIFICATION ERROR FUNCTION
# ============================================================

def classification_error(state):

    # Find selected feature positions
    selected_indices = np.where(state == 1)[0]

    # If no feature is selected
    if len(selected_indices) == 0:

        return 1.0


    # Get selected feature names
    selected_features = [
        feature_names[i]
        for i in selected_indices
    ]


    # Select only chosen features
    X_train_selected = X_train[
        selected_features
    ]

    X_test_selected = X_test[
        selected_features
    ]


    # Logistic Regression model
    model = Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )

    ])


    # Train model
    model.fit(
        X_train_selected,
        y_train
    )


    # Prediction
    prediction = model.predict(
        X_test_selected
    )


    # Accuracy
    accuracy = accuracy_score(
        y_test,
        prediction
    )


    # Classification error
    error = 1 - accuracy


    return error


# ============================================================
# 5. ENERGY FUNCTION
# ============================================================

def calculate_energy(state):

    # Number of selected features
    selected_features = np.sum(state)


    # Classification error
    error = classification_error(
        state
    )


    # Feature selection penalty
    feature_penalty = (
        0.02 *
        selected_features /
        len(state)
    )


    # Total energy
    energy = (
        error +
        feature_penalty
    )


    return energy


# ============================================================
# 6. CREATE NEIGHBOURING SOLUTION
# ============================================================

def create_neighbor(state):

    new_state = state.copy()


    # Select random feature
    index = np.random.randint(
        0,
        len(new_state)
    )


    # Flip feature
    # 0 -> 1
    # 1 -> 0
    new_state[index] = (
        1 - new_state[index]
    )


    # Make sure at least one feature remains
    if np.sum(new_state) == 0:

        new_state[index] = 1


    return new_state


# ============================================================
# 7. BOLTZMANN ANNEALING
# ============================================================

def boltzmann_machine(
    iterations=100,
    initial_temperature=1.0,
    cooling_rate=0.95
):


    # Random initial solution
    current_state = np.random.randint(
        0,
        2,
        len(feature_names)
    )


    # Ensure at least one feature
    if np.sum(current_state) == 0:

        current_state[
            np.random.randint(
                len(current_state)
            )
        ] = 1


    # Calculate initial energy
    current_energy = calculate_energy(
        current_state
    )


    # Best solution
    best_state = current_state.copy()

    best_energy = current_energy


    # Initial temperature
    temperature = initial_temperature


    # Store energy history
    energy_history = []


    # Annealing loop
    for iteration in range(iterations):


        # Generate neighbour
        new_state = create_neighbor(
            current_state
        )


        # Calculate new energy
        new_energy = calculate_energy(
            new_state
        )


        # Energy difference
        delta_energy = (
            new_energy -
            current_energy
        )


        # Accept better solution
        if delta_energy < 0:

            accept = True


        else:

            # Boltzmann probability
            probability = np.exp(
                -delta_energy /
                max(
                    temperature,
                    0.000001
                )
            )


            accept = (
                np.random.random()
                < probability
            )


        # Update current solution
        if accept:

            current_state = new_state

            current_energy = new_energy


        # Store best solution
        if current_energy < best_energy:

            best_state = (
                current_state.copy()
            )

            best_energy = (
                current_energy
            )


        # Store best energy
        energy_history.append(
            best_energy
        )


        # Reduce temperature
        temperature *= cooling_rate


    return (
        best_state,
        best_energy,
        energy_history
    )


# ============================================================
# 8. CAUCHY ANNEALING
# ============================================================

def cauchy_machine(
    iterations=100,
    initial_temperature=1.0,
    cooling_rate=0.95
):


    # Random initial solution
    current_state = np.random.randint(
        0,
        2,
        len(feature_names)
    )


    # Ensure at least one feature
    if np.sum(current_state) == 0:

        current_state[
            np.random.randint(
                len(current_state)
            )
        ] = 1


    # Initial energy
    current_energy = calculate_energy(
        current_state
    )


    # Best solution
    best_state = current_state.copy()

    best_energy = current_energy


    # Temperature
    temperature = initial_temperature


    # Energy history
    energy_history = []


    # Annealing loop
    for iteration in range(iterations):


        # Generate neighbour
        new_state = create_neighbor(
            current_state
        )


        # New energy
        new_energy = calculate_energy(
            new_state
        )


        # Energy difference
        delta_energy = (
            new_energy -
            current_energy
        )


        # Accept better solution
        if delta_energy < 0:

            accept = True


        else:

            # Cauchy probability
            probability = 1 / (
                1 +
                (
                    delta_energy /
                    max(
                        temperature,
                        0.000001
                    )
                ) ** 2
            )


            accept = (
                np.random.random()
                < probability
            )


        # Update current solution
        if accept:

            current_state = new_state

            current_energy = new_energy


        # Store best solution
        if current_energy < best_energy:

            best_state = (
                current_state.copy()
            )

            best_energy = (
                current_energy
            )


        # Store history
        energy_history.append(
            best_energy
        )


        # Cool temperature
        temperature *= cooling_rate


    return (
        best_state,
        best_energy,
        energy_history
    )


# ============================================================
# 9. DISPLAY SOLUTION
# ============================================================

def display_solution(
    name,
    state,
    energy
):


    # Selected feature indices
    selected_indices = np.where(
        state == 1
    )[0]


    # Selected feature names
    selected_features = [
        feature_names[i]
        for i in selected_indices
    ]


    # Classification error
    error = classification_error(
        state
    )


    # Accuracy
    accuracy = 1 - error


    print("\n")
    print("=" * 60)

    print(name)

    print("=" * 60)


    print("\nFeature Selection Vector:")

    print(state)


    print("\nSelected Features:")


    for feature in selected_features:

        print(
            "-",
            feature
        )


    print(
        "\nNumber of Selected Features:",
        len(selected_features)
    )


    print(
        "Classification Accuracy:",
        round(
            accuracy * 100,
            2
        ),
        "%"
    )


    print(
        "Classification Error:",
        round(
            error,
            4
        )
    )


    print(
        "Energy:",
        round(
            energy,
            4
        )
    )


# ============================================================
# OPTION 1
# BOLTZMANN MACHINE
# ============================================================

def option1():

    print("\nRunning Boltzmann Annealing...")

    (
        state,
        energy,
        history
    ) = boltzmann_machine(
        iterations=100
    )


    display_solution(
        "BOLTZMANN MACHINE RESULT",
        state,
        energy
    )


    # Plot energy
    plt.figure(
        figsize=(8, 5)
    )


    plt.plot(
        history
    )


    plt.xlabel(
        "Iteration"
    )


    plt.ylabel(
        "Best Energy"
    )


    plt.title(
        "Boltzmann Annealing Energy Optimization"
    )


    plt.grid(
        True
    )


    plt.show()


# ============================================================
# OPTION 2
# CAUCHY MACHINE
# ============================================================

def option2():

    print("\nRunning Cauchy Annealing...")

    (
        state,
        energy,
        history
    ) = cauchy_machine(
        iterations=100
    )


    display_solution(
        "CAUCHY MACHINE RESULT",
        state,
        energy
    )


    # Plot energy
    plt.figure(
        figsize=(8, 5)
    )


    plt.plot(
        history
    )


    plt.xlabel(
        "Iteration"
    )


    plt.ylabel(
        "Best Energy"
    )


    plt.title(
        "Cauchy Annealing Energy Optimization"
    )


    plt.grid(
        True
    )


    plt.show()


# ============================================================
# OPTION 3
# COMPARE BOLTZMANN AND CAUCHY
# ============================================================

def option3():

    print(
        "\nRunning Boltzmann Machine..."
    )


    (
        bm_state,
        bm_energy,
        bm_history
    ) = boltzmann_machine(
        iterations=100
    )


    print(
        "\nRunning Cauchy Machine..."
    )


    (
        cm_state,
        cm_energy,
        cm_history
    ) = cauchy_machine(
        iterations=100
    )


    print("\n")
    print("=" * 60)

    print(
        "BOLTZMANN VS CAUCHY COMPARISON"
    )

    print("=" * 60)


    print(
        "\nBoltzmann Energy:",
        round(
            bm_energy,
            4
        )
    )


    print(
        "Boltzmann Selected Features:",
        np.sum(
            bm_state
        )
    )


    print(
        "\nCauchy Energy:",
        round(
            cm_energy,
            4
        )
    )


    print(
        "Cauchy Selected Features:",
        np.sum(
            cm_state
        )
    )


    if bm_energy < cm_energy:

        print(
            "\nBetter Solution: Boltzmann Machine"
        )


    elif cm_energy < bm_energy:

        print(
            "\nBetter Solution: Cauchy Machine"
        )


    else:

        print(
            "\nBoth produced the same energy."
        )


    # Plot comparison
    plt.figure(
        figsize=(8, 5)
    )


    plt.plot(
        bm_history,
        label="Boltzmann"
    )


    plt.plot(
        cm_history,
        label="Cauchy"
    )


    plt.xlabel(
        "Iteration"
    )


    plt.ylabel(
        "Best Energy"
    )


    plt.title(
        "Boltzmann vs Cauchy Energy Optimization"
    )


    plt.legend()

    plt.grid(
        True
    )


    plt.show()


# ============================================================
# OPTION 4
# ENERGY FUNCTION ANALYZER
# ============================================================

def option4():

    print(
        "\n--- ENERGY FUNCTION ANALYZER ---"
    )


    # Select all features
    state = np.ones(
        len(feature_names),
        dtype=int
    )


    energy = calculate_energy(
        state
    )


    error = classification_error(
        state
    )


    print(
        "\nUsing all features:"
    )


    print(
        "Selected Features:",
        np.sum(state)
    )


    print(
        "Classification Error:",
        round(
            error,
            4
        )
    )


    print(
        "Classification Accuracy:",
        round(
            (1 - error) * 100,
            2
        ),
        "%"
    )


    print(
        "Total Energy:",
        round(
            energy,
            4
        )
    )


    # Single feature analysis
    print(
        "\nSingle Feature Energy:"
    )


    for i, feature in enumerate(
        feature_names
    ):


        state = np.zeros(
            len(feature_names),
            dtype=int
        )


        state[i] = 1


        energy = calculate_energy(
            state
        )


        print(
            feature,
            "-> Energy:",
            round(
                energy,
                4
            )
        )


# ============================================================
# OPTION 5
# RANDOM FEATURE SELECTION
# ============================================================

def option5():

    # Random binary feature vector
    state = np.random.randint(
        0,
        2,
        len(feature_names)
    )


    # Make sure at least one feature is selected
    if np.sum(state) == 0:

        state[
            np.random.randint(
                len(state)
            )
        ] = 1


    energy = calculate_energy(
        state
    )


    display_solution(
        "RANDOM FEATURE SELECTION",
        state,
        energy
    )


# ============================================================
# OPTION 6
# EXIT
# ============================================================

def option6():

    print(
        "\nExiting program..."
    )


# ============================================================
# MAIN MENU
# ============================================================

while True:

    print("\n")
    print("=" * 65)

    print(
        " BOLTZMANN AND CAUCHY FEATURE SELECTION SYSTEM "
    )

    print("=" * 65)


    print(
        "1. Boltzmann Machine Feature Selection"
    )


    print(
        "2. Cauchy Machine Feature Selection"
    )


    print(
        "3. Compare Boltzmann and Cauchy Machines"
    )


    print(
        "4. Energy Function Analyzer"
    )


    print(
        "5. Random Feature Selection"
    )


    print(
        "6. Exit"
    )


    choice = input(
        "\nEnter your choice: "
    )


    if choice == "1":

        option1()


    elif choice == "2":

        option2()


    elif choice == "3":

        option3()


    elif choice == "4":

        option4()


    elif choice == "5":

        option5()


    elif choice == "6":

        option6()

        break


    else:

        print(
            "\nInvalid choice."
        )

Dataset loaded successfully!

Dataset Shape:
(1143, 13)

Available Columns:
['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality', 'Id']

First 5 Rows:
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0       